In [ ]:
!pip install scikit-learn

In [ ]:
# ============================================================================
# Task 2 — Pragmatic Intent Detection (iSarcasm Benchmark)
# ============================================================================
# Phase 1: Binary Sarcasm Detection → F1 on sarcastic class
# Phase 2: Pragmatic Intent Category Classification → Macro F1 (6 labels)
# Overall Score: 0.5 * Detection F1 + 0.5 * Classification Macro F1
# Samples: 250 per phase (stratified, deterministic)
# ============================================================================

import kaggle_benchmarks as kbench
from dataclasses import dataclass
from sklearn.metrics import f1_score, classification_report
import pandas as pd
import os
import re
import json

# ============================================================================
# CONFIGURATION
# ============================================================================

DATASET_DIR = "/kaggle/input/datasets/tegzes/isarcasm"
CSV_FILENAME = "isarcasm2022.csv"

NUM_DETECT_SAMPLES = 250
NUM_CLASSIFY_SAMPLES = 250
RANDOM_SEED = 42

CATEGORY_COLS = [
    "sarcasm", "irony", "satire",
    "understatement", "overstatement", "rhetorical_question",
]

DETECT_LABELS = ["not_sarcastic", "sarcastic"]

DETECT_SYNONYMS = {
    "sarcastic": "sarcastic",
    "yes": "sarcastic",
    "true": "sarcastic",
    "1": "sarcastic",
    "not_sarcastic": "not_sarcastic",
    "not sarcastic": "not_sarcastic",
    "no": "not_sarcastic",
    "false": "not_sarcastic",
    "0": "not_sarcastic",
    "sincere": "not_sarcastic",
    "literal": "not_sarcastic",
    "genuine": "not_sarcastic",
}


# ============================================================================
# STRUCTURED OUTPUT SCHEMAS
# ============================================================================

@dataclass
class SarcasmDetection:
    """The LLM returns a sarcasm verdict."""
    verdict: str


@dataclass
class CategoryClassification:
    """The LLM returns applicable ironic speech categories."""
    categories: str


# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def find_csv_path():
    """Locate the CSV file in the dataset directory."""
    csv_path = os.path.join(DATASET_DIR, CSV_FILENAME)
    if os.path.exists(csv_path):
        return csv_path
    # Fallback: walk directory for any matching CSV
    for dirpath, _, filenames in os.walk(DATASET_DIR):
        for fname in filenames:
            if fname.lower() == CSV_FILENAME.lower():
                return os.path.join(dirpath, fname)
    # Last resort: any CSV
    for dirpath, _, filenames in os.walk(DATASET_DIR):
        for fname in filenames:
            if fname.endswith(".csv"):
                return os.path.join(dirpath, fname)
    raise FileNotFoundError(f"No CSV found under {DATASET_DIR}")


def safe_int(value):
    """Safely convert a CSV cell to 0 or 1."""
    if value is None:
        return 0
    value = str(value).strip()
    if value in ("", "nan", "NaN", "None"):
        return 0
    try:
        return int(float(value))
    except (ValueError, TypeError):
        return 0


def load_and_prepare_data():
    """Load iSarcasm CSV and prepare labels."""
    csv_path = find_csv_path()
    print(f"Loading dataset from: {csv_path}")

    df = pd.read_csv(csv_path)
    print(f"CSV columns: {list(df.columns)}")
    print(f"Total rows: {len(df)}")

    # Clean tweet text
    df["tweet"] = df["tweet"].astype(str).str.strip()
    df = df[df["tweet"].str.len() > 5].reset_index(drop=True)

    # Ensure binary sarcastic label
    df["sarcastic"] = df["sarcastic"].apply(safe_int)

    # Ensure category columns exist and are clean
    for col in CATEGORY_COLS:
        if col in df.columns:
            df[col] = df[col].apply(safe_int)
        else:
            df[col] = 0

    print(f"Sarcastic: {df['sarcastic'].sum()}, Non-sarcastic: {(df['sarcastic'] == 0).sum()}")
    return df


def select_stratified_detect_samples(df, n=NUM_DETECT_SAMPLES):
    """Stratified sample preserving sarcastic/non-sarcastic ratio."""
    sarc_pool = df[df["sarcastic"] == 1]
    non_pool = df[df["sarcastic"] == 0]

    actual_n = min(n, len(df))
    n_sarc = max(1, min(len(sarc_pool), round(actual_n * len(sarc_pool) / len(df))))
    n_non = min(len(non_pool), actual_n - n_sarc)
    n_sarc = min(len(sarc_pool), actual_n - n_non)

    sarc_selected = sarc_pool.sample(n=n_sarc, random_state=RANDOM_SEED)
    non_selected = non_pool.sample(n=n_non, random_state=RANDOM_SEED)

    combined = pd.concat([sarc_selected, non_selected]).sort_index().reset_index(drop=True)
    print(f"Detection sample: {n_sarc} sarcastic + {n_non} non-sarcastic = {len(combined)}")
    return combined


def select_sarcastic_samples(df, n=NUM_CLASSIFY_SAMPLES):
    """Deterministic subsample of sarcastic-only tweets."""
    sarc_pool = df[df["sarcastic"] == 1]
    actual_n = min(n, len(sarc_pool))
    selected = sarc_pool.sample(n=actual_n, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"Classification sample: {len(selected)} sarcastic tweets")
    return selected


def normalize_detect_response(raw):
    """Normalize LLM detection output to canonical label."""
    cleaned = raw.strip().lower()
    cleaned = re.sub(r"[^a-z0-9_ ]", "", cleaned)

    if cleaned in DETECT_SYNONYMS:
        return DETECT_SYNONYMS[cleaned]

    # Check "not sarcastic" before "sarcastic" to avoid substring match
    not_sarc_patterns = ["not.*sarcastic", "not_sarcastic", "sincere", "literal", "genuine"]
    for pat in not_sarc_patterns:
        if re.search(pat, cleaned):
            return "not_sarcastic"

    sarc_patterns = ["sarcastic", "sarcasm", "ironic", "irony"]
    for pat in sarc_patterns:
        if re.search(pat, cleaned):
            return "sarcastic"

    return "INVALID"


def parse_categories(raw):
    """Parse LLM category output into a 6-dim binary vector."""
    text = raw.strip().lower()
    vector = []
    for cat in CATEGORY_COLS:
        cat_variants = [cat, cat.replace("_", " ")]
        found = any(v in text for v in cat_variants)
        vector.append(1 if found else 0)
    return vector


def compute_category_macro_f1(gold_vectors, pred_vectors):
    """Compute macro F1 across the 6 category columns."""
    f1_scores = []
    for i, cat in enumerate(CATEGORY_COLS):
        golds_i = [g[i] for g in gold_vectors]
        preds_i = [p[i] for p in pred_vectors]
        f1_i = f1_score(golds_i, preds_i, average="binary", zero_division=0)
        f1_scores.append(f1_i)
    macro = sum(f1_scores) / len(f1_scores)
    return macro, dict(zip(CATEGORY_COLS, f1_scores))


# ============================================================================
# PROMPTS
# ============================================================================

def build_detect_prompt(tweet):
    return (
        "You are an expert in pragmatics and figurative language.\n\n"
        "Read the following tweet and determine whether the author intended "
        "it to be sarcastic. Sarcasm occurs when the surface meaning and the "
        "intended meaning diverge.\n\n"
        f"**Tweet:** {tweet}\n\n"
        "Respond with a JSON object containing one field:\n"
        '- "verdict": either "sarcastic" or "not_sarcastic"\n'
    )


def build_classify_prompt(tweet):
    return (
        "You are an expert in pragmatics, irony, and figurative language.\n\n"
        "The following tweet has been confirmed as sarcastic. Your task is to "
        "classify which type(s) of ironic speech it uses. A tweet may belong "
        "to multiple categories.\n\n"
        f"**Tweet:** {tweet}\n\n"
        "Categories:\n"
        "- sarcasm: contradicts the state of affairs and is critical toward an addressee\n"
        "- irony: contradicts the state of affairs but is NOT critical toward an addressee\n"
        "- satire: appears to support an addressee but contains underlying disagreement/mocking\n"
        "- understatement: undermines the importance of the state of affairs\n"
        "- overstatement: describes the state of affairs in obviously exaggerated terms\n"
        "- rhetorical_question: includes a question whose implied answer contradicts the state of affairs\n\n"
        "Respond with a JSON object containing one field:\n"
        '- "categories": a comma-separated string of applicable category names '
        '(e.g. "sarcasm, overstatement"). If none apply, use "none".\n'
    )


# ============================================================================
# MAIN BENCHMARK TASK
# ============================================================================

@kbench.task(
    name="isarcasm_pragmatic_intent_detection",
    description=(
        "Pragmatic intent detection on iSarcasm tweets. "
        "Phase 1: detect whether a tweet is sarcastic. "
        "Phase 2: classify the type(s) of ironic speech used. "
        "Overall score = 0.5 * Detection F1 + 0.5 * Classification Macro F1."
    ),
)
def pragmatic_intent_detection(llm) -> float:
    """
    Evaluate an LLM's ability to detect pragmatic intent — sarcasm detection
    and fine-grained ironic speech classification. Returns overall score 0.0-1.0.
    """

    df = load_and_prepare_data()

    # ==================================================================
    # PHASE 1: Binary Sarcasm Detection
    # ==================================================================
    detect_samples = select_stratified_detect_samples(df, NUM_DETECT_SAMPLES)

    y_true_detect = []
    y_pred_detect = []
    detect_log = []
    total_detect = len(detect_samples)

    for i, (idx, row) in enumerate(detect_samples.iterrows()):
        tweet = row["tweet"]
        gt = "sarcastic" if row["sarcastic"] == 1 else "not_sarcastic"

        prompt = build_detect_prompt(tweet)
        raw_verdict = ""

        with kbench.chats.new(f"detect_{i}"):
            try:
                prediction = llm.prompt(prompt, schema=SarcasmDetection)
                raw_verdict = prediction.verdict
            except Exception:
                with kbench.chats.new(f"detect_{i}_retry"):
                    raw_response = llm.prompt(prompt)
                    raw_verdict = raw_response.strip()
                    try:
                        parsed = json.loads(raw_response)
                        raw_verdict = parsed.get("verdict", raw_verdict)
                    except (json.JSONDecodeError, AttributeError):
                        pass

        pred = normalize_detect_response(str(raw_verdict))

        y_true_detect.append(gt)
        y_pred_detect.append(pred)

        status = "CORRECT" if pred == gt else "WRONG"
        print(f"[Detect {i+1}/{total_detect}] GT={gt} | Pred={pred} | {status}")

        detect_log.append({
            "sample": i + 1,
            "gt": gt,
            "pred": pred,
            "correct": pred == gt,
        })

    detect_f1 = f1_score(
        y_true_detect, y_pred_detect,
        labels=["sarcastic"], average="binary",
        pos_label="sarcastic", zero_division=0,
    )
    detect_correct = sum(r["correct"] for r in detect_log)
    detect_invalid = sum(1 for p in y_pred_detect if p == "INVALID")

    print(f"\n{'='*60}")
    print("PHASE 1: SARCASM DETECTION RESULTS")
    print(f"{'='*60}")
    print(f"Accuracy:    {detect_correct}/{total_detect} ({detect_correct/total_detect:.1%})")
    print(f"F1 (sarc):   {detect_f1:.4f}")
    print(f"Invalid:     {detect_invalid}/{total_detect}")
    if total_detect >= 4:
        print(classification_report(
            y_true_detect, y_pred_detect,
            labels=DETECT_LABELS, zero_division=0
        ))

    # ==================================================================
    # PHASE 2: Category Classification (sarcastic tweets only)
    # ==================================================================
    classify_samples = select_sarcastic_samples(df, NUM_CLASSIFY_SAMPLES)

    gold_vectors = []
    pred_vectors = []
    classify_log = []
    total_classify = len(classify_samples)

    for i, (idx, row) in enumerate(classify_samples.iterrows()):
        tweet = row["tweet"]
        gold_vec = [row[cat] for cat in CATEGORY_COLS]

        prompt = build_classify_prompt(tweet)
        raw_categories = ""

        with kbench.chats.new(f"classify_{i}"):
            try:
                prediction = llm.prompt(prompt, schema=CategoryClassification)
                raw_categories = prediction.categories
            except Exception:
                with kbench.chats.new(f"classify_{i}_retry"):
                    raw_response = llm.prompt(prompt)
                    raw_categories = raw_response.strip()
                    try:
                        parsed = json.loads(raw_response)
                        raw_categories = parsed.get("categories", raw_categories)
                    except (json.JSONDecodeError, AttributeError):
                        pass

        pred_vec = parse_categories(str(raw_categories))

        gold_vectors.append(gold_vec)
        pred_vectors.append(pred_vec)

        match = "EXACT" if gold_vec == pred_vec else "MISMATCH"
        print(f"[Classify {i+1}/{total_classify}] Gold={gold_vec} | Pred={pred_vec} | {match}")

        classify_log.append({
            "sample": i + 1,
            "gold": gold_vec,
            "pred": pred_vec,
            "exact_match": gold_vec == pred_vec,
        })

    classify_macro_f1, per_cat_f1 = compute_category_macro_f1(gold_vectors, pred_vectors)
    exact_matches = sum(r["exact_match"] for r in classify_log)

    print(f"\n{'='*60}")
    print("PHASE 2: CATEGORY CLASSIFICATION RESULTS")
    print(f"{'='*60}")
    print(f"Macro F1:        {classify_macro_f1:.4f}")
    print(f"Exact Match:     {exact_matches}/{total_classify} ({exact_matches/total_classify:.1%})")
    print(f"Per-category F1:")
    for cat, f1_val in per_cat_f1.items():
        print(f"  {cat:25s} {f1_val:.4f}")

    # ==================================================================
    # OVERALL SCORE
    # ==================================================================
    overall_score = 0.5 * detect_f1 + 0.5 * classify_macro_f1

    print(f"\n{'='*60}")
    print("OVERALL RESULTS")
    print(f"{'='*60}")
    print(f"Detection F1:        {detect_f1:.4f}")
    print(f"Classification F1:   {classify_macro_f1:.4f}")
    print(f"Overall Score:       {overall_score:.4f}")
    print(f"{'='*60}")

    # Assertions — report scores, always pass
    kbench.assertions.assert_true(
        0.0 <= overall_score <= 1.0,
        expectation=(
            f"Overall Score: {overall_score:.4f} "
            f"(Detection F1: {detect_f1:.4f}, "
            f"Classification Macro F1: {classify_macro_f1:.4f})"
        ),
    )

    kbench.assertions.assert_true(
        detect_invalid < total_detect,
        expectation="At least one detection prediction should be valid.",
    )

    return overall_score


# ============================================================================
# RUN
# ============================================================================

pragmatic_intent_detection.run(kbench.llm)